# 🚴 Cycling Data Management Assistant

This notebook provides a conversational interface for managing cycling data including riders, teams, and races.

In [1]:
import sys
from pathlib import Path

# Add parent directory to path so we can import from src
sys.path.insert(0, str(Path.cwd().parent))

# Now check we can import
try:
    from src.conversation.extractor import TaskExtractor
    print("✅ Imports working")
except ImportError as e:
    print(f"❌ Still can't import: {e}")
    print(f"Current directory: {Path.cwd()}")
    print(f"Looking for src in: {Path.cwd().parent}")

✅ Imports working


## Check Data File

First, let's verify your model.json file exists and check its structure:

In [2]:
import json
import os
from pathlib import Path

# CHANGE THIS PATH TO YOUR models.json LOCATION
# Options:
# data_file = Path('models.json')  # If in notebooks folder
# data_file = Path('../models.json')  # If in project root
# data_file = Path('/absolute/path/to/models.json')  # Absolute path

data_file = Path('models.json')  # <-- MODIFY THIS LINE TO YOUR FILE PATH

if not data_file.exists():
    print(f"❌ File not found at: {data_file.absolute()}")
    print(f"\nPlease update the data_file path in the cell above.")
    print(f"Current working directory: {Path.cwd()}")
else:
    file_size = os.path.getsize(data_file)
    print(f"✅ Found models.json")
    print(f"📊 File size: {file_size:,} bytes ({file_size/1024/1024:.1f} MB)")
    
    # Load and check the structure
    print("\n📋 Checking file structure...")
    print("Loading large file... this may take a few seconds...")
    
    try:
        with open(data_file, 'r') as f:
            data = json.load(f)  # Load the entire file properly
        
        print("\n✅ File loaded successfully!")
        print("\nTop-level keys found:")
        for key in data.keys():
            if isinstance(data[key], list):
                print(f"  - {key}: {len(data[key])} items")
            else:
                print(f"  - {key}: {type(data[key]).__name__}")
        
        # Show a sample of the data structure
        print("\n📝 Sample data structure:")
        if data.get('documentedRiders') and len(data['documentedRiders']) > 0:
            sample_rider = data['documentedRiders'][0]
            print("Sample documented rider fields:", list(sample_rider.keys())[:10])
        
        if data.get('documentedTeams') and len(data['documentedTeams']) > 0:
            sample_team = data['documentedTeams'][0]
            print("Sample documented team fields:", list(sample_team.keys())[:10])
            
    except json.JSONDecodeError as e:
        print(f"❌ Error parsing JSON: {e}")
        print("\nTrying to identify the issue...")
        
        # Try to identify where the JSON error is
        with open(data_file, 'r') as f:
            content = f.read(500)  # Read first 500 chars
            print("First 500 characters of file:")
            print(content)
            print("\n...")
            
    except Exception as e:
        print(f"❌ Unexpected error: {e}")

✅ Found models.json
📊 File size: 26,473,161 bytes (25.2 MB)

📋 Checking file structure...
Loading large file... this may take a few seconds...

✅ File loaded successfully!

Top-level keys found:
  - documentedRiders: 521 items
  - documentedTeams: 0 items
  - documentedRaces: 0 items
  - undocumentedRiders: 0 items
  - undocumentedTeams: 1895 items
  - undocumentedRaces: 6692 items

📝 Sample data structure:
Sample documented rider fields: ['id', 'firstName', 'lastName', 'birthDate', 'country', 'team', 'ranking', 'powerProfile', 'results']


## Initialize the Cycling System with Your Data

In [3]:
# Fix imports first
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

# Now run the debug test
from src.conversation.extractor import TaskExtractor
from src.core.connector_loader import load_connector

connector = load_connector('cycling')
extractor = TaskExtractor()

# Test with your exact query
query = "Find all french riders"
model_name = "llama3:8b"  # or whatever model you're using

print("Testing TaskExtractor directly...")
result = extractor.extract_task_details(query, model_name, connector)
print(f"Extracted action: '{result.get('action')}'")
print(f"Parameters: {result.get('parameters', {})}")

# Check if the action is in the connector
action = result.get('action')
if action in connector['actions']:
    print(f"✅ Action '{action}' is valid")
else:
    print(f"❌ Action '{action}' not found in connector")
    print(f"Available actions: {list(connector['actions'].keys())}")

Testing TaskExtractor directly...
Extracted action: 'find_rider'
Parameters: {'name_pattern': 'French', 'country': 'FR'}
✅ Action 'find_rider' is valid


## Launch the Chat Interface

Run the cell below to start the interactive chat interface:

In [5]:
# Display the interface
interface.display()

## Example Queries to Try

Here are some example queries you can try:

### Search Queries:
- "Find all French riders"
- "Search for riders in the top 100 ranking"
- "Find teams from France"
- "Show me races in July 2023"
- "Find riders with more than 500 points"

### Add Operations:
- "Add a new rider named John Doe from USA, born on 1995-03-15"
- "Create a new team called Quick-Step Alpha Vinyl from Belgium"
- "Add a new race called Paris-Nice in France starting on 2024-03-03"

### Modify Operations:
- "Modify rider 100033 to have 1000 points"
- "Add rider 98765 to team 32814"
- "Change team 32814's bike brand to Specialized"

### Document Management:
- "Document rider 98765" (moves from undocumented to documented)
- "Link rider 98765 to team 32814"

## Performance Check for Large File

Let's check how the system performs with your large file:

In [ ]:
import time

# Test search performance
print("⏱️ Testing search performance on large dataset...\n")

# Test 1: Search by country
start = time.time()
from src.domains.cycling.state_manager import CyclingStateManager
manager = CyclingStateManager(data_file)
results = manager.search_riders({'country': 'FR'})
elapsed = time.time() - start

print(f"Search for French riders:")
print(f"  Found {len(results)} riders in {elapsed:.3f} seconds")

# Test 2: Search with multiple criteria
start = time.time()
results = manager.search_riders({
    'min_rank': 1,
    'max_rank': 100,
    'documented': True
})
elapsed = time.time() - start

print(f"\nSearch for top 100 documented riders:")
print(f"  Found {len(results)} riders in {elapsed:.3f} seconds")

# Test 3: Team search
start = time.time()
teams = manager.search_teams({'country': 'FR'})
elapsed = time.time() - start

print(f"\nSearch for French teams:")
print(f"  Found {len(teams)} teams in {elapsed:.3f} seconds")

print("\n✅ Performance test complete")

## Data File Statistics

In [ ]:
# Show detailed statistics about your data file
print("📊 Detailed Data File Statistics:")
print("=" * 50)

with open(data_file, 'r') as f:
    data = json.load(f)

file_size = os.path.getsize(data_file)

print(f"File: {data_file.name}")
print(f"Size: {file_size:,} bytes ({file_size/1024/1024:.2f} MB)")
print(f"\nEntity counts:")
print(f"  Documented Riders: {len(data.get('documentedRiders', []))}")
print(f"  Documented Teams: {len(data.get('documentedTeams', []))}")
print(f"  Documented Races: {len(data.get('documentedRaces', []))}")
print(f"  Undocumented Riders: {len(data.get('undocumentedRiders', []))}")
print(f"  Undocumented Teams: {len(data.get('undocumentedTeams', []))}")
print(f"  Undocumented Races: {len(data.get('undocumentedRaces', []))}")

total_entities = sum([
    len(data.get('documentedRiders', [])),
    len(data.get('documentedTeams', [])),
    len(data.get('documentedRaces', [])),
    len(data.get('undocumentedRiders', [])),
    len(data.get('undocumentedTeams', [])),
    len(data.get('undocumentedRaces', []))
])

print(f"\nTotal entities: {total_entities:,}")
print(f"Average bytes per entity: {file_size/total_entities:.0f}")

# Sample some data
if data.get('documentedRiders'):
    print("\nSample rider:")
    rider = data['documentedRiders'][0]
    print(f"  Name: {rider.get('firstName', '')} {rider.get('lastName', '')}")
    print(f"  Country: {rider.get('country', 'N/A')}")
    if 'ranking' in rider:
        print(f"  Rank: {rider['ranking'].get('rank', 'N/A')}")
        print(f"  Points: {rider['ranking'].get('points', 'N/A')}")

## Backup Your Data (Optional)

Before making changes, you might want to create a backup:

In [ ]:
import shutil
from datetime import datetime

# Create a backup with timestamp
backup_name = f"models_backup_{datetime.now().strftime('%Y%m%d_%H%M%S')}.json"
backup_path = data_file.parent / backup_name

# Uncomment the line below to actually create the backup
# shutil.copy2(data_file, backup_path)
# print(f"✅ Backup created: {backup_path}")

print(f"To create a backup, uncomment the lines above")
print(f"Backup would be saved as: {backup_path}")